# 🧪 GRPO Submission - PGABL_Kendrick Filbert

## Group Relative Policy Optimization (GRPO) pada Model Fine-tuned

**Tujuan:** Menerapkan GRPO pada model yang telah di-fine-tune untuk mengajarkan model kemampuan *chain-of-thought reasoning* dengan format `<think>...</think>`.

### Reward Models:
1. `format_reward_func` - Reward shaping untuk format `<think>` tags
2. `reasoning_length_reward` - Reward proporsional berdasarkan panjang reasoning
3. `correctness_reward` - Reward berdasarkan kecocokan dengan ground truth (ROUGE)
4. `language_reward_func` - Reward/penalti berdasarkan bahasa (Indonesia vs Inggris)

## 1. Instalasi & Setup

In [ ]:
!pip uninstall vllm -y -q
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128 -q
!pip install --no-deps unsloth unsloth-zoo -q
!pip install "transformers>=4.51.3,<=5.5.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0" -q
!pip install peft accelerate bitsandbytes huggingface_hub sentencepiece protobuf -q
!pip install tyro hf_transfer cut_cross_entropy msgspec torchao -q
!pip install rouge-score langdetect wandb -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.4.8 requires bitsandbytes!=0.46.0,!=0.48.0,>=0.45.5, which is not installed.
unsloth 2026.4.8 requires hf_transfer, which is not installed.
unsloth 2026.4.8 requires tyro, which is not installed.
unsloth 2026.4.8 requires xformers>=0.0.27.post2; ("linux" in sys_platform or sys_platform == "win32") and (platform_machine == "AMD64" or platform_machine == "x86_64"), which is not installed.
unsloth-zoo 2026.4.9 requires cut_cross_entropy; python_version >= "3.10", which is not installed.
un

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Login HuggingFace
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ HuggingFace login berhasil")
except Exception as e:
    print(f"⚠️ HF login error: {e}")
    print("Masukkan token manual:")
    login()

# Wandb optional - skip jika token tidak cocok
try:
    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))
except:
    import wandb
    wandb.init(mode="disabled")
    print("⚠️ Wandb disabled - training tetap jalan tanpa logging")

from unsloth import FastLanguageModel
from datasets import load_dataset
import re
from rouge_score import rouge_scorer
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

print("✅ All imports successful!")

CUDA available: True
GPU: Tesla T4
✅ HuggingFace login berhasil


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kendrickf (kendrickf-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ All imports successful!


## 2. Load Model Hasil Fine-tuning

Memuat kembali model instruct/fine-tuned yang sudah dibuat di notebook sebelumnya.

In [ ]:
MODEL_NAME = "kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,  # Model fine-tuned Anda
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Apply LoRA for GRPO training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model fine-tuned loaded successfully!")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/173 [00:00<?, ?B/s]

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model fine-tuned loaded successfully!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 3. Persiapan Dataset untuk GRPO

In [ ]:
# Load dataset yang sama
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

# Untuk GRPO, kita perlu format yang memiliki prompt dan completion
# Ambil subset agar cukup untuk training di T4
dataset = dataset.shuffle(seed=42).select(range(min(2000, len(dataset))))

def format_for_grpo(examples):
    """Format dataset untuk GRPO training."""
    prompts = []
    completions = []

    for user_input, output in zip(examples["input"], examples["output"]):
        messages = [
            {"role": "system", "content": "Kamu adalah asisten AI yang membantu. Sebelum menjawab, pikirkan dulu langkah-langkahnya di dalam tag <think>...</think>, lalu berikan jawaban final."},
            {"role": "user", "content": str(user_input)},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)
        completions.append(str(output))

    return {"prompt": prompts, "completion": completions}

grpo_dataset = dataset.map(format_for_grpo, batched=True, batch_size=500, desc="Formatting for GRPO")
grpo_dataset = grpo_dataset.remove_columns([c for c in grpo_dataset.column_names if c not in ["prompt", "completion"]])

print(f"\n✅ GRPO Dataset size: {len(grpo_dataset)}")
print(f"\nContoh prompt:\n{grpo_dataset[0]['prompt'][:300]}...")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Formatting for GRPO:   0%|          | 0/2000 [00:00<?, ? examples/s]


✅ GRPO Dataset size: 2000

Contoh prompt:
<|im_start|>system
Kamu adalah asisten AI yang membantu. Sebelum menjawab, pikirkan dulu langkah-langkahnya di dalam tag <think>...</think>, lalu berikan jawaban final.<|im_end|>
<|im_start|>user
Tentukan hubungan antara variabel-variabel berikut.
Umur dan kecerdasan.<|im_end|>
<|im_start|>assistant...


## 4. Definisi Reward Models

### 4.1 format_reward_func
Reward shaping bertahap untuk format `<think>...</think>`:
- `<think>` di awal: +0.2
- `</think>` menutup: +0.3
- Format sempurna: +1.0
- Penalti tag ganda: -0.5

In [ ]:
def format_reward_func(completions, **kwargs):
    """
    Reward shaping untuk format <think>...</think>.

    Poin:
    - +0.2 jika ada tag <think> pembuka
    - +0.3 jika ada tag </think> penutup
    - +1.0 jika format sempurna (di awal, ditutup benar, diikuti jawaban)
    - -0.5 penalti jika tag muncul lebih dari satu kali (halusinasi)
    """
    rewards = []
    for completion in completions:
        reward = 0.0
        text = completion.strip() if isinstance(completion, str) else completion

        # Count tag occurrences
        think_open_count = text.count("<think>")
        think_close_count = text.count("</think>")

        # Penalti untuk tag yang muncul lebih dari sekali (halusinasi)
        if think_open_count > 1 or think_close_count > 1:
            reward = -0.5
            rewards.append(reward)
            continue

        # +0.2 jika ada tag <think> pembuka
        if "<think>" in text:
            reward += 0.2

        # +0.3 jika ada tag </think> penutup
        if "</think>" in text:
            reward += 0.3

        # +1.0 jika format sempurna:
        # - <think> di awal teks
        # - </think> menutup dengan benar
        # - Ada jawaban setelah </think>
        if text.startswith("<think>") and "</think>" in text:
            after_think = text.split("</think>", 1)
            if len(after_think) > 1 and after_think[1].strip():
                reward = 1.0  # Format sempurna, override poin bertahap

        rewards.append(reward)
    return rewards

# Test
test_cases = [
    "<think>Ini adalah proses berpikir</think> Ini jawaban final.",
    "<think>Proses berpikir</think>",  # Tidak ada jawaban setelahnya
    "Jawaban langsung tanpa think tag",
    "<think>A</think> jawaban <think>B</think>",  # Double tag
    "<think>Proses berpikir yang panjang tentang pertanyaan</think>Jawaban yang lengkap.",
]

print("Test format_reward_func:")
print("=" * 60)
for tc in test_cases:
    r = format_reward_func([tc])
    print(f"  Input: {tc[:60]}...")
    print(f"  Reward: {r[0]}")
    print()

Test format_reward_func:
  Input: <think>Ini adalah proses berpikir</think> Ini jawaban final....
  Reward: 1.0

  Input: <think>Proses berpikir</think>...
  Reward: 0.5

  Input: Jawaban langsung tanpa think tag...
  Reward: 0.0

  Input: <think>A</think> jawaban <think>B</think>...
  Reward: -0.5

  Input: <think>Proses berpikir yang panjang tentang pertanyaan</thin...
  Reward: 1.0



### 4.2 reasoning_length_reward
Reward proporsional berdasarkan panjang reasoning di dalam `<think>`:
- Tidak ada/kosong: +0.0
- < 50 karakter: +0.2
- 50-199 karakter: +0.5
- ≥ 200 karakter: +1.0

In [ ]:
def reasoning_length_reward(completions, **kwargs):
    """
    Reward berdasarkan panjang teks di dalam <think>...</think>.
    Toleran jika proses berpikir terpotong oleh batas maksimal token.

    Poin:
    - +0.0: Tidak ada tag <think> atau isinya kosong/spasi
    - +0.2: Panjang < 50 karakter
    - +0.5: Panjang 50-199 karakter
    - +1.0: Panjang >= 200 karakter
    """
    rewards = []
    for completion in completions:
        text = completion.strip() if isinstance(completion, str) else completion

        # Cari konten di dalam <think>...</think>
        # Juga handle kasus terpotong (ada <think> tapi tidak ada </think>)
        if "<think>" not in text:
            rewards.append(0.0)
            continue

        # Extract konten thinking
        if "</think>" in text:
            # Normal case: tag lengkap
            match = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
            thinking_content = match.group(1).strip() if match else ""
        else:
            # Toleran: tag terpotong (truncated oleh max token)
            thinking_content = text.split("<think>", 1)[1].strip()

        # Hitung panjang
        content_length = len(thinking_content)

        if content_length == 0:
            rewards.append(0.0)
        elif content_length < 50:
            rewards.append(0.2)
        elif content_length < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)

    return rewards

# Test
test_cases_length = [
    "Jawaban tanpa think",
    "<think></think> Jawaban",
    "<think>singkat</think> Jawaban",
    "<think>" + "x" * 100 + "</think> Jawaban dengan reasoning sedang",
    "<think>" + "Ini adalah reasoning yang sangat panjang. " * 15 + "</think> Jawaban detail.",
    "<think>Reasoning yang terpotong karena max token limit dan tidak ada closing tag",
]

print("Test reasoning_length_reward:")
print("=" * 60)
for tc in test_cases_length:
    r = reasoning_length_reward([tc])
    # Get thinking length
    if "<think>" in tc and "</think>" in tc:
        m = re.search(r"<think>(.*?)</think>", tc, re.DOTALL)
        tlen = len(m.group(1).strip()) if m else 0
    elif "<think>" in tc:
        tlen = len(tc.split("<think>", 1)[1].strip())
    else:
        tlen = 0
    print(f"  Thinking length: {tlen:>4d} chars -> Reward: {r[0]}")

Test reasoning_length_reward:
  Thinking length:    0 chars -> Reward: 0.0
  Thinking length:    0 chars -> Reward: 0.0
  Thinking length:    7 chars -> Reward: 0.2
  Thinking length:  100 chars -> Reward: 0.5
  Thinking length:  629 chars -> Reward: 1.0
  Thinking length:   73 chars -> Reward: 0.5


### 4.3 correctness_reward
Reward berdasarkan kecocokan output model dengan ground truth menggunakan ROUGE score.

In [ ]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def correctness_reward(completions, completion=None, **kwargs):
    """
    Reward berdasarkan kemiripan output akhir model dengan ground truth.
    Menggunakan ROUGE-L score.

    Poin:
    - +1.0 jika output mengandung ground truth atau ROUGE-L >= 0.5
    - Skor ROUGE-L proporsional untuk yang lainnya
    """
    # Ground truth dari dataset
    ground_truths = kwargs.get("completion", completion)
    if ground_truths is None:
        return [0.0] * len(completions)

    rewards = []
    for comp, gt in zip(completions, ground_truths):
        text = comp.strip() if isinstance(comp, str) else comp

        # Ambil jawaban setelah </think> (jika ada)
        if "</think>" in text:
            final_answer = text.split("</think>", 1)[1].strip()
        else:
            final_answer = text

        # Cek apakah ground truth terkandung dalam output
        if gt.strip().lower() in final_answer.lower():
            rewards.append(1.0)
            continue

        # Hitung ROUGE-L score
        score = scorer.score(gt.strip(), final_answer)
        rouge_l = score['rougeL'].fmeasure

        if rouge_l >= 0.5:
            rewards.append(1.0)
        else:
            rewards.append(rouge_l)

    return rewards

# Test
print("Test correctness_reward:")
test_comp = ["<think>Berpikir...</think> Kurangi, gunakan kembali, daur ulang"]
test_gt = ["Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."]
r = correctness_reward(test_comp, completion=test_gt)
print(f"  Reward: {r[0]:.4f}")

Test correctness_reward:
  Reward: 1.0000


### 4.4 language_reward_func
Penalti untuk bahasa Inggris, reward untuk bahasa Indonesia.

In [ ]:
from langdetect import detect, LangDetectException

def language_reward_func(completions, **kwargs):
    """
    Reward berdasarkan bahasa output.

    Poin:
    - +1.0 jika output murni bahasa Indonesia
    - -0.5 jika output menggunakan bahasa Inggris
    """
    rewards = []
    for completion in completions:
        text = completion.strip() if isinstance(completion, str) else completion

        # Ambil jawaban setelah </think> (jika ada)
        if "</think>" in text:
            final_answer = text.split("</think>", 1)[1].strip()
        else:
            final_answer = text

        # Skip jika terlalu pendek untuk deteksi bahasa
        if len(final_answer) < 10:
            rewards.append(0.0)
            continue

        try:
            lang = detect(final_answer)
            if lang == 'id':
                rewards.append(1.0)   # Bahasa Indonesia
            elif lang == 'en':
                rewards.append(-0.5)  # Penalti bahasa Inggris
            else:
                rewards.append(0.0)   # Bahasa lain (netral)
        except LangDetectException:
            rewards.append(0.0)

    return rewards

# Test
test_lang = [
    "<think>Berpikir</think> Ini adalah jawaban dalam bahasa Indonesia yang benar.",
    "<think>Thinking</think> This is an answer in English language.",
    "<think>Pikir</think> ok",
]

print("Test language_reward_func:")
print("=" * 60)
for tc in test_lang:
    r = language_reward_func([tc])
    print(f"  Input: {tc[:60]}...")
    print(f"  Reward: {r[0]}")
    print()

Test language_reward_func:
  Input: <think>Berpikir</think> Ini adalah jawaban dalam bahasa Indo...
  Reward: 1.0

  Input: <think>Thinking</think> This is an answer in English languag...
  Reward: -0.5

  Input: <think>Pikir</think> ok...
  Reward: 0.0



## 5. Training GRPO

Menggunakan `GRPOTrainer` dari TRL dan Unsloth dengan parameter yang disesuaikan untuk T4 GPU.

In [ ]:
# Konfigurasi GRPO - disesuaikan untuk T4 GPU (16GB VRAM)
grpo_config = GRPOConfig(
    # Output
    output_dir="outputs_grpo",
    run_name="grpo_reasoning",
    report_to="wandb",

    # Training parameters
    learning_rate=5e-6,
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # GRPO specific - PENTING untuk mitigasi OOM
    num_generations=2,           # Jumlah generasi per prompt (kecil untuk hemat VRAM)
    max_completion_length=256,   # Batas panjang completion (mitigasi OOM)
    max_prompt_length=512,       # Batas panjang prompt

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Logging
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,

    seed=42,
)

print("GRPO Configuration:")
print(f"  num_generations       : {grpo_config.num_generations}")
print(f"  max_completion_length : {grpo_config.max_completion_length}")
print(f"  max_prompt_length     : {grpo_config.max_prompt_length}")
print(f"  learning_rate         : {grpo_config.learning_rate}")
print(f"  max_steps             : {grpo_config.max_steps}")

GRPO Configuration:
  num_generations       : 2
  max_completion_length : 256
  max_prompt_length     : 512
  learning_rate         : 5e-06
  max_steps             : 100


In [ ]:
# Inisialisasi GRPOTrainer dengan semua reward functions
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
)

print("✅ GRPOTrainer initialized with 4 reward functions")
print("  1. format_reward_func")
print("  2. reasoning_length_reward")
print("  3. correctness_reward")
print("  4. language_reward_func")

✅ GRPOTrainer initialized with 4 reward functions
  1. format_reward_func
  2. reasoning_length_reward
  3. correctness_reward
  4. language_reward_func


In [11]:
# Jalankan GRPO Training
print("🚀 Memulai GRPO Training...")
print("=" * 50)
print("⚠️  Proses ini memakan waktu. Monitor VRAM usage.")
print()

trainer.train()

print("\n✅ GRPO Training selesai!")

🚀 Memulai GRPO Training...
⚠️  Proses ini memakan waktu. Monitor VRAM usage.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
10,0.000000,0.950000,0.070711,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.950000,0.100000
15,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
20,0.000000,1.045000,0.106066,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,-0.015000,0.070000,0.060000,0.120000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.800000,0.070711,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000174,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.800000,0.273205
30,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
10,0.000000,0.950000,0.070711,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.950000,0.100000
15,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
20,0.000000,1.045000,0.106066,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000013,-0.015000,0.070000,0.060000,0.120000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.800000,0.070711,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000174,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.800000,0.273205
30,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
35,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
40,0.000000,1.025000,0.247487,249.550000,230.200000,256.000000,0.950000,25.400000,25.400000,25.400000,0.000016,0.075000,0.150000,0.075000,0.150000,0.000000,0.000000,0.875000,0.250000
45,0.000000,1.000000,0.000000,250.700000,234.800000,256.000000,0.950000,30.000000,30.000000,30.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
50,0.000000,1.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000017,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000


Unsloth: Restored added_tokens_decoder metadata in outputs_grpo/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_grpo/checkpoint-100/tokenizer_config.json.



✅ GRPO Training selesai!


## 6. Push Model GRPO ke Hugging Face

In [12]:
# Push model GRPO
HF_USERNAME = "kendrickfff"
GRPO_MODEL_NAME = f"{HF_USERNAME}/Qwen2.5-1.5B-Indonesian-Assistant-GRPO"

print(f"📤 Mengunggah model GRPO ke: {GRPO_MODEL_NAME}")

model.push_to_hub_merged(
    GRPO_MODEL_NAME,
    tokenizer,
    save_method="merged_16bit",
)

print(f"\n✅ Model GRPO berhasil diunggah!")

# Update link file
with open("link_huggingface.txt", "w") as f:
    f.write(f"Fine-tuned Model: https://huggingface.co/{HF_USERNAME}/Qwen2.5-1.5B-Indonesian-Assistant\n")
    f.write(f"GRPO Model: https://huggingface.co/{GRPO_MODEL_NAME}\n")

📤 Mengunggah model GRPO ke: kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant-GRPO


Unsloth: Restored added_tokens_decoder metadata in kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant-GRPO/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...stant-GRPO/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt-GRPO/model.safetensors:   3%|3         |  104MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:15<00:00, 75.41s/it]


Unsloth: Merge process complete. Saved to `/content/kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant-GRPO`

✅ Model GRPO berhasil diunggah!


## 7. Test Model GRPO

Testing model GRPO dengan pertanyaan yang ditentukan untuk memverifikasi kemampuan reasoning.

In [13]:
# Test model GRPO
FastLanguageModel.for_inference(model)

test_prompt = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

messages = [
    {"role": "system", "content": "Kamu adalah asisten AI hukum yang membantu menjawab pertanyaan berdasarkan peraturan ketenagakerjaan Indonesia. Sebelum menjawab, pikirkan dulu langkah-langkahnya di dalam tag <think>...</think>, lalu berikan jawaban final."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("=" * 70)
print("TEST CASE WAJIB - Model GRPO dengan Reasoning")
print("=" * 70)
print(f"\n📝 Prompt: {test_prompt}")
print(f"\n🤖 Response:\n{response}")
print("\n" + "=" * 70)

# Verifikasi format
if "<think>" in response:
    print("✅ Tag <think> terdeteksi - Model menunjukkan proses reasoning!")
else:
    print("⚠️  Tag <think> tidak terdeteksi dalam response")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


TEST CASE WAJIB - Model GRPO dengan Reasoning

📝 Prompt: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

🤖 Response:
Saya tidak bisa memberikan jawaban yang pasti karena informasi tentang peraturan ketenagakerjaan Indonesia dapat bervariasi dari negara. Namun, umumnya, perusahaan yang melibatkan tenaga kerja harus membayar lembur yang terjadi dalam satu bulan atau dalam satu tahun dengan mengikuti ketentuan yang berlaku. Untuk mengidentifikasi apakah Anda berhak untuk uang lembur, perlu dicari peraturan yang berlaku terkait dengan lembur dan apakah Anda memenuhi syarat untuk mendapatkan uang lembur. Penting untuk melihat aturan ketenagakerjaan Anda sendiri dan mengikuti persyaratan untuk lembur.spNet
spNet
<think>Saya perlu mengikuti persyaratan ketenagakerjaan saya sendiri untuk menentukan apakah saya berhak untuk uang lembur. Apakah saya perlu menghubungi pengaturan ketenagakerjaan untuk memeriksa syaratnya? Apakah ada persyaratan y

## ✅ GRPO Selesai!

### Ringkasan:
- ✅ Model fine-tuned dimuat kembali untuk GRPO
- ✅ 4 Reward Models diimplementasikan:
  - `format_reward_func`: Reward shaping bertahap untuk format `<think>`
  - `reasoning_length_reward`: Reward proporsional berdasarkan panjang reasoning
  - `correctness_reward`: Reward berdasarkan ROUGE-L dengan ground truth
  - `language_reward_func`: Reward bahasa Indonesia, penalti bahasa Inggris
- ✅ GRPOTrainer dijalankan dengan mitigasi OOM (num_generations=2, max_completion_length=256)
- ✅ Test case wajib dijalankan menunjukkan proses reasoning `<think>`
- ✅ Model diunggah ke Hugging Face